In [2]:
# Construct the commulant expansion of the term
# assuming that all correlations are already captured by the lower order terms, the cumulant expansion of an expectation values <X1 X2 ... Xn>_c = 0, so that we can expand <X1 X2 ... Xn> into
# <X1 X2 ... Xn> = \sum_{p in P(I)} (|p|-1)! (-1)^{|p|} \prod_{B in p} <\prod_{i \in B} X_i>
# where |p| is the number of terms in the partition and P(I) is the set of all partitions of the set I = {1, 2, ..., n},
# and B is a block in the partition p (i.e. a subset of I), with the i in B the indices of the operators in the block B
# (notice, that the operators don't get reordered)
# Example: <X1 X2 X3> = <X1 X2><X3> + <X1 X3><X2> + <X2 X3><X1> - 2<X1><X2><X3>
include("operator_terms.jl")
using Combinatorics
using LaTeXStrings

In [3]:
# Get the number of operators in a term
function how_many_operators_in_term(term)::Tuple{Int, Int, Int}
    # Returns for a term:
    # total_op: total number of operators
    # ferm_op: number of fermionic operators
    # bos_op: number of bosonic operators
    ferm_op = length(term.spin_types)
    bos_op = 0
    exponents = term.exponents
    for i in 1:length(exponents)
        bos_op += exponents[i]
    end
    return ferm_op + bos_op, ferm_op, bos_op
end
# Test
term = make_term("++-xi*yj")
how_many_operators_in_term(term)

(5, 2, 3)

In [4]:
# a function that constructs all combinations of numbers (1 to n-1) that sum up to n
# only combinations from large to small (decreasing but not strictly decreasing)
function combinations_summing_to_n(n::Int, max_i::Int=-1)::Vector{Vector{Int}}
    # which combinations of numbers sum up to n
    # for example, for n=3, we have [[1, 1, 1], [2, 1], [3]] # from large to small, to avoid double counting
    # n is the number we want to sum up to
    # max_i is the smallest number to the right (as we want the numbers to decrease it sets a limit to following numbers)
    # returns a Vector of Vectors, where each Vector is a combination of numbers that sum up to n (Vector{Vector{Int}})
    all_combinations::Vector{Vector{Int}} = Vector{Vector{Int}}()
    if max_i == -1
        max_i = n
    elseif max_i > n
        max_i = n
    end
    remaining::Int = n
    curr_combination::Array{Int, 1} = Array{Int, 1}()
    for i in 1:max_i
        remaining = n - i
        if remaining == 0
            push!(all_combinations, [i])
        else
            # get all combinations of remaining numbers
            remaining_combinations = combinations_summing_to_n(remaining, i)
            for j in 1:length(remaining_combinations)
                curr_combination = [i]
                append!(curr_combination, remaining_combinations[j])
                push!(all_combinations, curr_combination)
            end
        end
    end
    return all_combinations
end
# Test
combinations_summing_to_n(3)

3-element Vector{Vector{Int}}:
 [1, 1, 1]
 [2, 1]
 [3]

In [5]:
# a function that returns modified combinations_summing_to_n,
# if multiple elements have the same value, we group them in a sublist
# for example, for n=3, we turn [[1, 1, 1], [2, 1], [3]] into [[3, 1]], [[1,2], [1,1]], [[1,3]]]   ([counts, value]])
# or [2,1,1] into [[1,2], [2, 1]]
# as each expectation value is a scalar, their order does not matter, and we don't wish to double count,
# so we group them, and then generate their value combinations for the complete groups, to ensure single ocunting
function grouped_combinations_summing_to_n(n::Int, max_i::Int=-1)::Vector{Vector{Tuple{Int, Int}}}
    all_combinations::Vector{Vector{Int}} = combinations_summing_to_n(n, max_i)[1:end-1]   # remove last one - all in one
    # group the combinations
    grouped_combinations::Vector{Vector{Tuple{Int, Int}}} = Vector{Vector{Tuple{Int, Int}}}()
    curr_comb::Vector{Tuple{Int, Int}} = Tuple{Int, Int}[]
    for comb in all_combinations
        # check if any of the unique values in comb appear more than once
        # get unique values and their counts
        unique_values = sort(unique(comb), rev=true)
        counts = [count(x->x==i, comb) for i in unique_values]
        # if all counts are 1, then we don't need to group
        curr_comb = Tuple{Int, Int}[]
        for (val, counts) in zip(unique_values, counts)
            push!(curr_comb, (counts, val))
        end
        push!(grouped_combinations, curr_comb)
    end
    return grouped_combinations
end
# Test
grouped_combinations_summing_to_n(6)

10-element Vector{Vector{Tuple{Int, Int}}}:
 [(6, 1)]
 [(1, 2), (4, 1)]
 [(2, 2), (2, 1)]
 [(3, 2)]
 [(1, 3), (3, 1)]
 [(1, 3), (1, 2), (1, 1)]
 [(2, 3)]
 [(1, 4), (2, 1)]
 [(1, 4), (1, 2)]
 [(1, 5), (1, 1)]

In [6]:
### no longer in use  --- wrong results for comulants (script is correct but not needed)
function array_split(arr::Vector{Int}, n::Int)::Vector{Vector{Int}}   # equivalent to numpys array split
    if length(arr)%n != 0
        error("Array length not divisible by n")
    end
    k::Int = Int(length(arr)/n)
    return [arr[(i-1)*n+1:(i*n)] for i in 1:k]
end
# Test
array_split([1,2,3,4,5,6], 3)

2-element Vector{Vector{Int}}:
 [1, 2, 3]
 [4, 5, 6]

In [7]:
# A function that splits n elements into k equally sized groups, 
# but in all combinations of splits, where different orders of the groups are not counted as different 
# and different orders within the groups also do not count as different
function all_equal_array_splits(arr::Vector{Int}, size_of_boxes::Int)::Vector{Vector{Vector{Int}}}
    n::Int = length(arr)
    if n%size_of_boxes != 0
        error("Array length not divisible by size_of_boxes")
    end
    number_of_boxes::Int = Int(n/size_of_boxes)  # number of boxes
    if number_of_boxes == 1
        return [[arr]]
    end
    # get all combinations of numbers that sum up to number_of_boxes
    # first box starts with first elements, second box starts with second remaining element, etc.
    curr_boxing::Vector{Vector{Int}} = Vector{Vector{Int}}()
    all_boxings::Vector{Vector{Vector{Int}}} = Vector{Vector{Vector{Int}}}()
    curr_box::Vector{Int} = Vector{Int}()
    anti_array::Vector{Int} = Vector{Int}()
    for comb in combinations(arr[2:end], size_of_boxes-1)
        curr_box = [arr[1]]
        append!(curr_box, comb)
        anti_array = setdiff(arr, curr_box)
        subsequent_boxes::Vector{Vector{Vector{Int}}} = all_equal_array_splits(anti_array, size_of_boxes)
        for sub_box in subsequent_boxes
            curr_boxing =  Vector{Vector{Int}}()
            push!(curr_boxing, curr_box)
            for sub in sub_box
                push!(curr_boxing, sub)
            end
            push!(all_boxings, curr_boxing)
        end
    end
    return all_boxings
end
# Test
all_equal_array_splits(collect(1:4), 2)

3-element Vector{Vector{Vector{Int}}}:
 [[1, 2], [3, 4]]
 [[1, 3], [2, 4]]
 [[1, 4], [2, 3]]

In [8]:
# A function that partitions the numbers from 1 to n into all possible partitions (<X1 X2><X3> -> function output is [[1, 2], [3]], for <X1 X2 X3> -> function output is [[1, 2, 3]] and <X1><X2><X3> -> function output is [[1], [2], [3]])
function partitions(n_array::Vector{Int}, comb::Union{Vector{Vector{Tuple{Int, Int}}}, Nothing}=nothing)::Vector{Vector{Vector{Int}}}
    n = length(n_array)
    if isa(comb, Nothing)
        comb = grouped_combinations_summing_to_n(n, n)
    end
    all_partitions::Vector{Vector{Vector{Int}}} = Vector{Vector{Vector{Int}}}()
    new_partition::Vector{Vector{Int}} = Vector{Vector{Int}}()
    distributed_partitions::Vector{Vector{Vector{Int}}} = Vector{Vector{Vector{Int}}}()
    for c in comb
        c0 = c[1]
        for partition in combinations(n_array, c0[1]*c0[2])
            # get anti partition for remaining elements
            distributed_partitions = all_equal_array_splits(partition, c0[2])
            if length(c) > 1
                # get anti partition for remaining elements
                remaining_elements = setdiff(n_array, partition)
                remaining_partition = partitions(remaining_elements, [c[2:end]])
                for p in remaining_partition
                    # concatenate distributed_partition and p into new_partition
                    for i in 1:length(distributed_partitions)
                        new_partition = deepcopy(distributed_partitions[i])
                        for i in 1:length(p)
                            push!(new_partition, p[i])
                        end
                        push!(all_partitions, new_partition)
                    end
                end
            else
                for partition in distributed_partitions
                    push!(all_partitions, partition)
                end
            end
        end
    end
    return all_partitions
end
# Test
n = 6
part = partitions(collect(1:n))

202-element Vector{Vector{Vector{Int}}}:
 [[1], [2], [3], [4], [5], [6]]
 [[1, 2], [3], [4], [5], [6]]
 [[1, 3], [2], [4], [5], [6]]
 [[1, 4], [2], [3], [5], [6]]
 [[1, 5], [2], [3], [4], [6]]
 [[1, 6], [2], [3], [4], [5]]
 [[2, 3], [1], [4], [5], [6]]
 [[2, 4], [1], [3], [5], [6]]
 [[2, 5], [1], [3], [4], [6]]
 [[2, 6], [1], [3], [4], [5]]
 [[3, 4], [1], [2], [5], [6]]
 [[3, 5], [1], [2], [4], [6]]
 [[3, 6], [1], [2], [4], [5]]
 ⋮
 [[1, 4, 5, 6], [2, 3]]
 [[2, 3, 4, 5], [1, 6]]
 [[2, 3, 4, 6], [1, 5]]
 [[2, 3, 5, 6], [1, 4]]
 [[2, 4, 5, 6], [1, 3]]
 [[3, 4, 5, 6], [1, 2]]
 [[1, 2, 3, 4, 5], [6]]
 [[1, 2, 3, 4, 6], [5]]
 [[1, 2, 3, 5, 6], [4]]
 [[1, 2, 4, 5, 6], [3]]
 [[1, 3, 4, 5, 6], [2]]
 [[2, 3, 4, 5, 6], [1]]

In [9]:
mutable struct Cumulant
    partitions::Vector{Vector{Vector{Int}}}   # cumulant terms 
    weights::Vector{Int}                      # Weights of the cumulant terms 
    operator::Vector{Int}                     # The operator for which we expanded the cumulant
end

function subscript_int(num::Int)::String
    subscript_dict::Dict{Char, String} = Dict('1'=>"₁", '2'=>"₂", '3'=>"₃", '4'=>"₄", '5'=>"₅", '6'=>"₆", '7'=>"₇", '8'=>"₈", '9'=>"₉", '0'=>"₀", '-'=>"₋", '+'=>"₊")
    num_str::String = string(num)
    sub_str::String = ""
    for c in num_str
        sub_str *= subscript_dict[c]
    end
    return sub_str
end

function cumulant_braket_to_str(t::Vector{Int}; do_latex::Bool=true, operator_names::Union{String, Vector{String}}="\\hat{X}")::String
    term_str::String = ""
    name_is_string::Bool = isa(operator_names, String)
    for i in 1:length(t)
        if name_is_string
            if do_latex
                term_str *= operator_names*"_"*string(t[i])
            else
                term_str *= operator_names*subscript_int(t[i])
            end
        else
            term_str *= operator_names[t[i]]
        end
    end
    if do_latex
        term_str = "\\braket{"*term_str*"}"
    else
        term_str = "〈"*term_str*"〉"
        #term_str = "⟨"*term_str*"⟩"
    end
    return term_str
end

function cumulant_term_to_str(term::Vector{Vector{Int}}, weight::Int; do_latex::Bool=true, operator_names::Union{String, Vector{String}}="\\hat{X}")::String
    str::String = ""
    name_is_string::Bool = isa(operator_names, String)
    if !name_is_string
        cum_length = 0
        for t in term
            cum_length += length(t)
        end
        if length(operator_names) < cum_length
            error("Length of operator_names does not match number of operators in term")
        end
    end
    if weight >= 0
        str *= "+"
    else
        str *= "-"
    end
    if abs(weight) != 1
        str *= string(abs(weight))
    end
    for t in term
        str *= cumulant_braket_to_str(t, do_latex=do_latex, operator_names=operator_names)
    end
    return str
end

function cumulant_to_str(cumulant::Cumulant; do_latex::Bool=true, operator_names::Union{String, Vector{String}}="\\hat{X}")::String
    str::String = ""
    curr_str::String = ""
    str = cumulant_braket_to_str(cumulant.operator, do_latex=do_latex, operator_names=operator_names) * " = "
    for i in 1:length(cumulant.partitions)
        curr_str = cumulant_term_to_str(cumulant.partitions[i], cumulant.weights[i], do_latex=do_latex, operator_names=operator_names)
        if i == 1 && curr_str[1] == '+'
            str *= " "*curr_str[2:end]
        else 
            if curr_str[1] == '+'
                str *= " + "*curr_str[2:end]
            else
                str *= " - "*curr_str[2:end]
            end
        end
    end
    return str
end

function Base.show(io::IO, ::MIME"text/plain", cumulant::Cumulant)
    str = cumulant_to_str(cumulant, do_latex=false)
    print(io, str)
end

function Base.show(io::IO, ::MIME"text/latex", cumulant::Cumulant)
    str = latexstring(cumulant_to_str(cumulant, do_latex=true))
    print(io, str)
end


In [10]:
mutable struct Cumulant_indexed
    partitions::Vector{Vector{Vector{Int}}}   # cumulant terms 
    weights::Vector{Int}                      # Weights of the cumulant terms 
    operator::Vector{Int}                     # The operator for which we expanded the cumulant
end
# Same as Cumulant, but different name, to make sure that the indexes have been replaced with those for the apropriate operators


In [11]:
function sort_by_length_and_content(A::Vector{Vector{Int}})
    # sort by length of lectors and by content
    # for example, [ [2,3], [2], [1,2], [1]]
    # becomes [ [1,2], [2,3], [1], [2]]
    # A is a vector of vectors
    # returns a vector of vectors
    A = sort(A)
    return sort(A, by=x->(-length(x), x))
end
# Test
#sort_by_length_and_content([[2,3], [2], [1,2], [1]])
# a function that orders terms in a cumulant by the order of the terms, finds terms that are the same and removes such doubles
function order_cumulant(cumulant)
    # first sort terms in cumulant by the number of operators in the term
    partitions::Vector{Vector{Vector{Int}}} = cumulant.partitions
    weights::Vector{Int} = cumulant.weights
    for i in 1:length(partitions)
        # sort by length or content
        partitions[i] = sort_by_length_and_content(partitions[i])
    end
    inds = sortperm(partitions)
    partitions = partitions[inds]
    weights = weights[inds]
    # create length vector so that [[1,2], [1]] ->[2,1]
    lengths::Vector{Vector{Int}} = [[length(term) for term in partition] for partition in partitions]
    inds = sortperm(lengths, rev=true)
    partitions = partitions[inds]
    weights = weights[inds]
    # sort partitions by length
    inds = sortperm(partitions, by=length)
    partitions = partitions[inds]
    weights = weights[inds] 
    return Cumulant(partitions, weights, cumulant.operator)
end

function remove_doubles(cumulant) # also remove zeros
    # check if adjacent vectors are the same 
    cumulant = order_cumulant(cumulant)
    partitions::Vector{Vector{Vector{Int}}} = cumulant.partitions
    weights::Vector{Int} = cumulant.weights 
    new_partitions::Vector{Vector{Vector{Int}}} = Vector{Vector{Vector{Int}}}()
    new_weights::Vector{Int} = Vector{Int}()
    # then remove doubles
    i = 1
    counter = 1
    curr_partition = partitions[counter]
    curr_weight = weights[counter]
    while i < length(partitions)
        if curr_partition == partitions[i+1]
            curr_weight += weights[i+1]
        else
            if curr_weight != 0
                push!(new_partitions, curr_partition)
                push!(new_weights, curr_weight)
            end
            curr_partition = partitions[i+1]
            curr_weight = weights[i+1]
        end
        i += 1
    end
    if curr_weight != 0
        push!(new_partitions, curr_partition)
        push!(new_weights, curr_weight)
    end
    return Cumulant(new_partitions, new_weights, cumulant.operator)
end

function mrange(lengths::Vector{Int})
    curr_index::Vector{Int} = ones(Int, length(lengths))
    max_iterations = prod(lengths)
    counter = 0
    chnl = Channel() do channel
        while counter < max_iterations
            put!(channel, copy(curr_index))
            counter += 1
            for i in length(lengths):-1:1
                curr_index[i] += 1
                if curr_index[i] > lengths[i]
                    curr_index[i] = 1
                    if i == 1
                        return
                    end
                else
                    break
                end
            end
        end
    end
    return chnl
end
# Test 
#cumulant = cumulant_expansion(collect(1:6), do_order=false) # not yet defined. see below
##display(cumulant)
#display(group_terms_in_cumulant(cumulant)) 

mrange (generic function with 1 method)

In [12]:
function cumulant_expansion(n_array::Vector{Int}; do_order::Bool=true)::Cumulant
    # Returns partitions and a vector of their weights, to construct the cumulant expansion
    # n_array is an array of numbers from 1 to n, where n is the number of operators in the term
    # returns a tuple of partitions and their weights
    # partitions is a vector of vectors of vectors, where each vector is a partition of the numbers from 1 to n
    # weights is a vector of integers, where each integer is the weight of the corresponding partition
    # the weight of a partition is (-1)^|p| (|p|-1)! where |p| is the number of terms in the partition
    # for example, for <X1 X2 X3> = <X1 X2><X3> + <X1 X3><X2> + <X2 X3><X1> - 2<X1><X2><X3>
    # we have partitions = [[[1, 2], [3]], [[1, 3], [2]], [[2, 3], [1]], [[1], [2], [3]]]
    # and weights = [1, 1, 1, -2]
    all_partitions::Vector{Vector{Vector{Int}}} = partitions(n_array)
    lengths::Vector{Int} = [length(p) for p in all_partitions]
    weights::Vector{Int} = [(-1)^l * factorial(l-1) for l in lengths]
    cumulant = Cumulant(all_partitions, weights, n_array)
    if do_order
        cumulant = order_cumulant(cumulant)
    end
    return cumulant
end
# Test
cumulant = cumulant_expansion([1,2,3,4])


〈\hat{X}₁\hat{X}₂\hat{X}₃\hat{X}₄〉 =  〈\hat{X}₁\hat{X}₂\hat{X}₃〉〈\hat{X}₄〉 + 〈\hat{X}₁\hat{X}₂\hat{X}₄〉〈\hat{X}₃〉 + 〈\hat{X}₁\hat{X}₃\hat{X}₄〉〈\hat{X}₂〉 + 〈\hat{X}₂\hat{X}₃\hat{X}₄〉〈\hat{X}₁〉 + 〈\hat{X}₁\hat{X}₂〉〈\hat{X}₃\hat{X}₄〉 + 〈\hat{X}₁\hat{X}₃〉〈\hat{X}₂\hat{X}₄〉 + 〈\hat{X}₁\hat{X}₄〉〈\hat{X}₂\hat{X}₃〉 - 2〈\hat{X}₁\hat{X}₂〉〈\hat{X}₃〉〈\hat{X}₄〉 - 2〈\hat{X}₁\hat{X}₃〉〈\hat{X}₂〉〈\hat{X}₄〉 - 2〈\hat{X}₁\hat{X}₄〉〈\hat{X}₂〉〈\hat{X}₃〉 - 2〈\hat{X}₂\hat{X}₃〉〈\hat{X}₁〉〈\hat{X}₄〉 - 2〈\hat{X}₂\hat{X}₄〉〈\hat{X}₁〉〈\hat{X}₃〉 - 2〈\hat{X}₃\hat{X}₄〉〈\hat{X}₁〉〈\hat{X}₂〉 + 6〈\hat{X}₁〉〈\hat{X}₂〉〈\hat{X}₃〉〈\hat{X}₄〉

In [13]:
function n_th_order_cumulant(n::Int)::Cumulant
    arr::Vector{Int} = [i for i in 1:n]
    return cumulant_expansion(arr)
end
# Test
n_th_order_cumulant(3)

〈\hat{X}₁\hat{X}₂\hat{X}₃〉 =  〈\hat{X}₁\hat{X}₂〉〈\hat{X}₃〉 + 〈\hat{X}₁\hat{X}₃〉〈\hat{X}₂〉 + 〈\hat{X}₂\hat{X}₃〉〈\hat{X}₁〉 - 2〈\hat{X}₁〉〈\hat{X}₂〉〈\hat{X}₃〉

In [14]:
#### Scaling
for n in 2:6
    println(n, " & ", length(partitions(collect(1:n))), "  \\\\")
end

2 & 1  \\
3 & 4  \\
4 & 14  \\
5 & 51  \\
6 & 202  \\


In [15]:
# a function that finds partitions of same type in a cumulant (same number of terms, and same lengths of terms), groups them together and makes a list
# i.e. [[1,2], [3]] and [[1,3], [2]] are of the same type, and should be grouped together as [2,1] doe to their lengths 
function group_terms_in_cumulant(cumulant, remove_empty::Bool=false)
    partitions = cumulant.partitions
    weights = cumulant.weights
    type_dict::Dict{Vector{Int}, Tuple{Int, Int}} = Dict{Vector{Int}, Tuple{Int, Int}}() # contains weight and number of terms
    # for every partition in partitions define type, check if type exists in dictionary and potentially add to dictionary
    function partition2type(partition)
        # returns a vector of integers, where each integer is the length of the corresponding term in the partition
        return [length(term) for term in partition]
    end
    for (p, w) in zip(partitions, weights)
        p_type = partition2type(p)
        if haskey(type_dict, p_type)
            type_dict[p_type] = (type_dict[p_type][1], type_dict[p_type][2] + 1)
        else
            type_dict[p_type] = (w, 1)
        end
    end
    if remove_empty
        # remove empty terms
        for key in keys(type_dict)
            if type_dict[key][1] == 0
                delete!(type_dict, key)
            end
        end
    end
    return type_dict
end
# Test
#cumulant = cumulant_expansion(collect(1:4))
#display(cumulant)
#group_terms_in_cumulant(cumulant)

group_terms_in_cumulant (generic function with 2 methods)

In [16]:
function reduce_partition(cumulant::Cumulant, m, lower_cumulants::Vector{Cumulant})
    lengths_lower_cumulants = [length(cumulant.partitions) for cumulant in lower_cumulants]
    curr_partitions::Vector{Vector{Vector{Int}}} = cumulant.partitions
    curr_weights::Vector{Int} = cumulant.weights
    new_partitions = Vector{Vector{Vector{Int}}}()
    new_weights = Vector{Int}()
    for i in 1:length(curr_partitions)
        # is any element of curr_partitions[i] of length > m
        any_larger = false
        bigger_than_m = Vector{Int}(undef, length(curr_partitions[i]))
        which_inds = Vector{Int}()
        how_long = Vector{Int}(undef, length(curr_partitions[i]))
        for j in 1:length(curr_partitions[i])
            curr_len = length(curr_partitions[i][j])
            if curr_len > m
                bigger_than_m[j] = curr_len - m
                how_long[j] = lengths_lower_cumulants[bigger_than_m[j]]
                push!(which_inds, j)
                any_larger = true
            else
                bigger_than_m[j] = 0
                how_long[j] = 1
            end
        end
        which_inds = reverse(which_inds)
        if any_larger
            for inds in mrange(how_long)
                new_element = deepcopy(curr_partitions[i])
                new_weight = deepcopy(curr_weights[i])

                for ind in which_inds
                    curr_inds = new_element[ind]
                    curr_partition_inds = lower_cumulants[bigger_than_m[ind]].partitions[inds[ind]]
                    curr_partition_weights = lower_cumulants[bigger_than_m[ind]].weights[inds[ind]]
                    # use the indexes in curr_partition_inds to replace the indexes in curr_inds
                    inserter = [curr_inds[curr_partition_inds[i]] for i in 1:length(curr_partition_inds)]
                    splice!(new_element, ind, inserter)
                    new_weight *= curr_partition_weights
                end
                push!(new_partitions, new_element)
                push!(new_weights, new_weight)
            end
        else
            push!(new_partitions, curr_partitions[i])
            push!(new_weights, cumulant.weights[i])
        end
    end
    # sort the elements
    # sort partitions by length
    inds = sortperm(new_partitions, by=length)
    new_partitions = new_partitions[inds]
    new_weights = new_weights[inds]
    for i in 1:length(new_partitions)
        # sort by length or content
        new_partitions[i] = sort_by_length_and_content(new_partitions[i])
    end
    new_cumulant = Cumulant(new_partitions, new_weights, cumulant.operator)
    return remove_doubles(new_cumulant)
end
# Test
#m = 2
#lower_cumulants = [ cumulant_expansion(collect(1:i)) for i in [m+1]]
#cumulant = cumulant_expansion(collect(1:4))
#display(cumulant)
#reduced_cumulant = reduce_partition(cumulant, m, lower_cumulants)
#display(reduced_cumulant)

reduce_partition (generic function with 1 method)

In [22]:
# Experiment: 
# Reduce terms via cumulants until every term is at most of order m 
function expand_to_m(n, m=2)
    diff = n - m  # how often we must reduce the cumulant
    if diff < 0
        error("m must be smaller than the number of operators")
    end
    if n == m+1
        return cumulant_expansion(collect(1:n))
    end
    n_array = collect(1:n)
    cumulant = cumulant_expansion(n_array)
    lower_cumulants = [ cumulant_expansion(collect(1:i)) for i in m+1:n ]
    lengths_lower_cumulants = Vector{Int}(undef, length(lower_cumulants))
    lengths_lower_cumulants[1] = length(lower_cumulants[1].partitions)
    # find terms in lower order cumulants that are of order > m  and reduce them using the lower order cumulants
    for i in 2:diff
        # use reduce_partition to reduce the cumulant
        lower_cumulants[i] = reduce_partition(lower_cumulants[i], m, lower_cumulants[1:i-1])
        lengths_lower_cumulants[i] = length(lower_cumulants[i].partitions)
    end
    # reduce the cumulant
    return lower_cumulants[end]
end
# Test 
m = 2
n = 4
cumulant = cumulant_expansion(collect(1:n))
display(group_terms_in_cumulant(cumulant)) 
reduced_cumulant = expand_to_m(n, m)
display(group_terms_in_cumulant(reduced_cumulant, true))

Dict{Vector{Int}, Tuple{Int, Int}} with 4 entries:
  [2, 1, 1]    => (-2, 6)
  [3, 1]       => (1, 4)
  [2, 2]       => (1, 3)
  [1, 1, 1, 1] => (6, 1)

Dict{Vector{Int}, Tuple{Int, Int}} with 2 entries:
  [2, 2]       => (1, 3)
  [1, 1, 1, 1] => (-2, 1)

In [30]:
display(cumulant_expansion(collect(1:n)))
display(reduced_cumulant)